## Load data

### Load plaintexts

In [ ]:
# Initialize an array to store the extracted values
hex_arrays = []
path="/media/abish/Extreme Pro/Sbox_600/"#sbox_600/"#"courses/sca101/res/"
# Open the file and process each line
with open(path+"plaintexts.txt", "r") as file:
    for line in file:
        # Split the line by spaces and extract the hex values (ignoring the label before the colon)
        parts = line.strip().split(":")
        if len(parts) == 2:
            hex_values = parts[1].strip().split()
            hex_arrays.append(hex_values)

# Print the resulting array
## for idx, hex_array in enu|merate(hex_arrays, start=1):
   ## print(f"vmem{idx}: {hex_array}")

### Load Trace data

In [ ]:
import numpy as np
import os

# Directory containing the data files
data_directory = path+"Result"
number_of_traces =600 #422
# Initialize a list to store all the numpy arrays
data_arrays = []

# Iterate through all 422 files
for i in range(1, number_of_traces+1):  # From 1 to 422
    file_name = f"plot_data_check_{i}.data"
    file_path = os.path.join(data_directory, file_name)

    # Initialize a list to store the second values for the current file
    second_values = []

    # Read the file
    with open(file_path, "r") as file:
        for line in file:
            # Ignore comment lines starting with '#'
            if line.startswith("#"):
                continue

            # Split the line into two values
            parts = line.strip().split()
            if len(parts) == 2:
                # Extract the second value
                second_values.append(float(parts[1]))

    # Convert the list of second values to a NumPy array and store it
    data_arrays.append(np.array(second_values))

# Now `data_arrays` contains 422 NumPy arrays, each corresponding to one file.
# Example: Print the shape of the first array to verify
print(f"First array shape: {np.shape(data_arrays)}")
print(f"First array shape: {data_arrays[1].shape}")
##print(f"First array content: {data_arrays[0]}")


### Convert data + remove last value of traces

In [ ]:
import numpy as np
import os
trace_arrayx= np.array(data_arrays)[:, :-1] #remove last value of traces
number_of_traces =600
textin_arraycc=hex_arrays[:number_of_traces]
textin_array = [[int(value, 16) for value in hex_array] for hex_array in textin_arraycc] # convert to hex
print(f"textin_array array shape: {np.shape(textin_array)}")
print(f"trace_arrayx array shape: {np.shape(trace_arrayx)}")

## CPA

In [ ]:
from tqdm.notebook import trange
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
from Crypto.Cipher import AES  # PyCryptodome library for AES verification
import time


sbox = [
    # 0    1    2    3    4    5    6    7    8    9    a    b    c    d    e    f
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76, # 0
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0, # 1
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15, # 2
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75, # 3
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84, # 4
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf, # 5
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8, # 6
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2, # 7
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73, # 8
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb, # 9
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79, # a
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08, # b
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a, # c
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e, # d
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf, # e
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16  # f
]

def aes_internal(inputdata, key):
    return sbox[inputdata ^ key]

HW = [bin(n).count("1") for n in range(0, 256)]


def mean(X):
    return np.sum(X, axis=0)/len(X)

def std_dev(X, X_bar):
    return np.sqrt(np.sum((X-X_bar)**2, axis=0))

def cov(X, X_bar, Y, Y_bar):
    return np.sum((X-X_bar)*(Y-Y_bar), axis=0)
# Function to compute KL Divergence for normal distributions
def kl_divergence(mu_x, sigma_x, mu_y, sigma_y):
    return ((mu_x - mu_y)**2) / (2 * sigma_y**2) + (sigma_x**2) / (2 * sigma_y**2) - 0.5 + np.log(sigma_y / sigma_x)


# Correct key for comparison
correct_key = [0x2b, 0x7e, 0x15, 0x16, 0x28, 0xae, 0xd2, 0xa6, 0xab, 0xf7, 0x15, 0x88, 0x09, 0xcf, 0x4f, 0x3c]


t_bar = np.sum(trace_arrayx, axis=0)/len(trace_arrayx)
o_t = np.sqrt(np.sum((trace_arrayx - t_bar)**2, axis=0))

cparefs = [0] * 16 #put your key byte guess correlations here
bestguess = [0] * 16 #put your key byte guesses here
startt= time.time()
for bnum in trange(0, 16):
    maxcpa = [0] * 256
    for kguess in range(0, 256):
        hws = np.array([[HW[aes_internal(textin[bnum],kguess)] for textin in textin_array]]).transpose()
        hws_bar = mean(hws)
        o_hws = std_dev(hws, hws_bar)
        correlation = cov(trace_arrayx, t_bar, hws, hws_bar)
        cpaoutput = correlation/(o_t*o_hws)
        maxcpa[kguess] = max(abs(cpaoutput))
    bestguess[bnum] = np.argmax(maxcpa)
    cparefs[bnum] = max(maxcpa)
print("Best Key Guess: ", end="")
for b in bestguess: print("%02x " % b, end="")
print("\n", cparefs)
endd= time.time()
elapsed_time = endd - startt
print(f"Elapsed time: {elapsed_time:.5f} seconds")
if bestguess == correct_key:
        print("Correct key found!")
else:
    differences = [i for i, (a, b) in enumerate(zip(bestguess, correct_key)) if a != b]
    print(f"Correct key not found! Differences at indices: {differences}")
    for i in differences:
        print(f"Index {i}: bestguess = {bestguess[i]}, correct_key = {correct_key[i]}")

## Find Leakage Time interval, and Leakage model

In [ ]:
from scipy.stats import pearsonr
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import trange
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
from Crypto.Cipher import AES  # PyCryptodome library for AES verification
import time


sbox = [
    # 0    1    2    3    4    5    6    7    8    9    a    b    c    d    e    f
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76, # 0
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0, # 1
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15, # 2
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75, # 3
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84, # 4
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf, # 5
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8, # 6
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2, # 7
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73, # 8
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb, # 9
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79, # a
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08, # b
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a, # c
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e, # d
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf, # e
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16  # f
]

def aes_internal(inputdata, key):
    return sbox[inputdata ^ key]

HW = [bin(n).count("1") for n in range(0, 256)]


def mean(X):
    return np.sum(X, axis=0)/len(X)

def std_dev(X, X_bar):
    return np.sqrt(np.sum((X-X_bar)**2, axis=0))

def cov(X, X_bar, Y, Y_bar):
    return np.sum((X-X_bar)*(Y-Y_bar), axis=0)
# Function to compute KL Divergence for normal distributions
def kl_divergence(mu_x, sigma_x, mu_y, sigma_y):
    return ((mu_x - mu_y)**2) / (2 * sigma_y**2) + (sigma_x**2) / (2 * sigma_y**2) - 0.5 + np.log(sigma_y / sigma_x)


# Correct key for comparison
correct_key = [0x2b, 0x7e, 0x15, 0x16, 0x28, 0xae, 0xd2, 0xa6, 0xab, 0xf7, 0x15, 0x88, 0x09, 0xcf, 0x4f, 0x3c]


cparefs = [0] * 16 #put your key byte guess correlations here
bestguess = [0] * 16 #put your key byte guesses here
selected_byte = 0 # Define a byte
threshold = 0.105  # Define a threshold
# Example Usage
key_guess = correct_key[selected_byte]  # Example key guess
print(f"correct key is: {key_guess}, and byte is {selected_byte}")
textin_array_np = np.array(textin_array)
plaintexts = textin_array_np[:, int(selected_byte)]


def extract_leakage_model(textin_array, key_guess):
    leakage_model = []
    for pt in textin_array:
        # Compute input to S-box
        sbox_input = pt ^ key_guess

        # S-box output
        sbox_output = sbox[sbox_input]

        # Extract the first bit (MSB)
        first_bit = (sbox_output >> 7) & 1

        # Convert to +1/-1
        leakage_model.append(1 if first_bit == 1 else -1)
    return np.array(leakage_model)

# Compute correlation over time
def compute_correlation(trace_array, leakage_model):
    correlation_over_time = []
    for t in range(trace_array.shape[1]):  # Loop over time samples
        trace_at_time = trace_array[:, t]
        corr, _ = pearsonr(trace_at_time, leakage_model)
        correlation_over_time.append(abs(corr))  # Absolute correlation
    return np.array(correlation_over_time)

# Find significant leakage time intervals
def find_leakage_time_interval(correlation_over_time, threshold=0.1):
    return np.where(correlation_over_time > threshold)[0]


# Extract leakage model
leakage_model = extract_leakage_model(plaintexts, key_guess)

# Compute correlation over time
##correlation_over_time = compute_correlation(trace_arrayx, leakage_model)

# Plot correlation over time
##plt.plot(correlation_over_time)
##plt.title("Correlation Over Time for First Bit Leakage Model")
##plt.xlabel("Time Samples")
##plt.ylabel("Correlation")
##plt.show()

# Identify leakage time interval

##leakage_time_indices = find_leakage_time_interval(correlation_over_time, threshold=threshold)
##print(f"Leakage Time Indices: {leakage_time_indices}")

In [ ]:

import numpy as np

# AES S-box (as provided in your snippet)
from scipy.stats import pearsonr
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import trange
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
from Crypto.Cipher import AES  # PyCryptodome library for AES verification
import time
import numpy as np
from scipy.stats import pearsonr, norm


sbox = [
    # 0    1    2    3    4    5    6    7    8    9    a    b    c    d    e    f
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76, # 0
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0, # 1
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15, # 2
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75, # 3
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84, # 4
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf, # 5
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8, # 6
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2, # 7
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73, # 8
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb, # 9
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79, # a
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08, # b
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a, # c
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e, # d
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf, # e
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16  # f
]

def aes_internal(inputdata, key):
    return sbox[inputdata ^ key]

HW = [bin(n).count("1") for n in range(0, 256)]


def mean(X):
    return np.sum(X, axis=0)/len(X)

def std_dev(X, X_bar):
    return np.sqrt(np.sum((X-X_bar)**2, axis=0))

def cov(X, X_bar, Y, Y_bar):
    return np.sum((X-X_bar)*(Y-Y_bar), axis=0)
# Function to compute KL Divergence for normal distributions
def kl_divergence(mu_x, sigma_x, mu_y, sigma_y):
    return ((mu_x - mu_y)**2) / (2 * sigma_y**2) + (sigma_x**2) / (2 * sigma_y**2) - 0.5 + np.log(sigma_y / sigma_x)


def pearson_confidence_interval(x, y, alpha=0.05):
    # Calculate Pearson's correlation coefficient using scipy.stats.pearsonr
    r, p_value = pearsonr(x, y)
    n = len(x)

    # Fisher's z-transform to approximate normality
    z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)

    # Calculate the critical z-value
    z_critical = norm.ppf(1 - alpha / 2)

    # Compute confidence interval in z-space
    z_interval = z + np.array([-1, 1]) * z_critical * se

    # Transform back to correlation scale
    r_interval = np.tanh(z_interval)

    return r, r_interval, p_value

###############################################################################
# Single-bit HD model extraction
###############################################################################
def extract_leakage_model_hd_bit(plaintexts, key_byte, selected_bit=7):
    """
    Compute the single-bit HD between the input to the first-round S-box
    and its output for a chosen bit.

    :param plaintexts: 1D array/list of plaintext bytes (for selected byte index).
    :param key_byte:    The correct key byte for that position.
    :param selected_bit: Bit index to check (7 = MSB, 0 = LSB).
    :return:           1D numpy array with HD values (-1 or 1) for each plaintext.
    """
    leakage_model = []
    for pt in plaintexts:
        # 1) Previous value: input to the first S-box (pt XOR key)
        sbox_in = pt ^ key_byte
        prev_bit = (sbox_in >> selected_bit) & 0x1

        # 2) Current value: S-box output
        sbox_out = sbox[sbox_in]
        curr_bit = (sbox_out >> selected_bit) & 0x1

        # 3) Single-bit Hamming Distance = XOR of the bits
        hd_bit = prev_bit ^ curr_bit
        leakage_model.append(1 if hd_bit == 1 else -1)

    return np.array(leakage_model)
# Compute correlation over time
def compute_correlation(trace_array, leakage_model):
    correlation_over_time = []
    for t in range(trace_array.shape[1]):  # Loop over time samples
        trace_at_time = trace_array[:, t]
        corr, _ = pearsonr(trace_at_time, leakage_model)
        correlation_over_time.append(corr)
    return np.array(correlation_over_time)

# Find significant leakage time intervals
def find_leakage_time_interval(correlation_over_time, threshold=0.1):
    return np.where(abs(correlation_over_time) > threshold)[0]

###############################################################################
# Example usage
###############################################################################
if __name__ == "__main__":
    tistart= time.time()
    selected_byte = 0 # Define a byte
    chosen_bit = 0  # or any bit in [0..7]
    threshold = 0.105#0.110  # Define a threshold
    corkey = [0x2b, 0x7e, 0x15, 0x16, 0x28, 0xae, 0xd2, 0xa6, 0xab, 0xf7, 0x15, 0x88, 0x09, 0xcf, 0x4f, 0x3c]
    key_guess = corkey[selected_byte]
    NTR=600
    print(f"correct key is: {key_guess}, and byte is {selected_byte}")
    textin_array_np = np.array(textin_array)
    plaintexts = textin_array_np[:, int(selected_byte)]
    # Generate random plaintext bytes for demonstration
    # Choose which bit to analyze (7 = MSB, 0 = LSB)


    # Extract single-bit HD leakage model
    hd_model = extract_leakage_model_hd_bit(
        plaintexts,
        key_guess,
        selected_bit=chosen_bit
    )
    print(f"HD model is: {len(hd_model)}")
    # Now, 'hd_model' is a 1D array of length N containing {0,1}
    #  - 0 means no bit flip from S-box input to output for that bit
    #  - 1 means the bit flipped from S-box input to output
    print("HD Model for Single Bit (0 or 1) =", hd_model)
    # Compute correlation over time
    correlation_over_time = compute_correlation(trace_arrayx, hd_model[0:NTR])
    print(f"timr is: {time.time()-tistart}")
    plt.plot(correlation_over_time)
    plt.title("Correlation Over Time for First Bit Leakage Model")
    plt.xlabel("Time Samples")
    plt.ylabel("Correlation")
    plt.show()
    # Identify leakage time interval
    num_time_steps = trace_arrayx.shape[1]

# Initialize an array to hold the correlation values
    correlations = np.zeros(num_time_steps)

# Calculate correlation at each time step
    for t in range(num_time_steps):
    # Extract the power measurements at time t (shape: (422,))
        P_t = trace_arrayx[:, t]

    # --- Method 1: Using np.corrcoef ----------------------------------
        #correlations[t] = np.corrcoef(L, P_t)[0, 1]

    # --- (Alternative) Method 2: Using the definition of covariance ---
        cov_L_Pt = np.mean((hd_model[0:NTR] - np.mean(hd_model[0:NTR])) * (P_t - np.mean(P_t)))
        std_L = np.std(hd_model[0:NTR], ddof=1)
        std_Pt = np.std(P_t, ddof=1)
        correlations[t] = cov_L_Pt / (std_L * std_Pt)

# Plot the correlation values
    plt.plot(correlations)
    plt.xlabel('Time Index')
    plt.ylabel('Correlation')
    plt.title('Correlation between Leakage Model and Power Traces over Time')
    plt.grid(True)
    plt.show()
    leakage_time_indices = find_leakage_time_interval(correlation_over_time, threshold=threshold)
    print(f"Leakage Time Indices: {leakage_time_indices}")


In [ ]:
leakage_model = hd_model

print(leakage_model)

### Leakage time calculate

In [ ]:
import numpy as np
import os

# Directory containing the data files
data_directory = path

# Initialize a list to store all the numpy arrays
data_arraystime = []

# Iterate through all 422 files

file_name = f"Result/plot_data_check_1.data"
file_path = os.path.join(data_directory, file_name)

    # Initialize a list to store the second values for the current file
first_values = []

    # Read the file
with open(file_path, "r") as file:
    for line in file:
            # Ignore comment lines starting with '#'
        if line.startswith("#"):
            continue

            # Split the line into two values
        parts = line.strip().split()
        if len(parts) == 2:
                # Extract the second value
            first_values.append(float(parts[0]))

    # Convert the list of second values to a NumPy array and store it
data_arraystime.append(np.array(first_values))

# Now `data_arrays` contains 422 NumPy arrays, each corresponding to one file.
# Example: Print the shape of the first array to verify
print(f"First array shape: {np.shape(data_arraystime)}")
print(f"First array shape: {data_arraystime[0].shape}")

time_start = data_arraystime[0][int(min(leakage_time_indices))]  # Start time in ns
time_end = data_arraystime[0][int(max(leakage_time_indices))]  # End time in ns

print(f"time idx = {int(min(leakage_time_indices))}, time_start: {time_start}")
print(f"time idx = {int(max(leakage_time_indices))}, time_end: {time_end}")


In [ ]:
import os
import re
import warnings
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
import matplotlib.pyplot as plt  # for plotting (optional)

###############################################################################
# 1) Single-file parser
###############################################################################
def parse_vcd_signals_per_clock(vcd_file, clock_signal_name):
    """
    Parse a single VCD file, capturing integer values of all signals
    at each rising edge of the given clock signal.
    Returns a dict: {signal_name: [val_per_clock, ...], ...}
    """
    signal_map       = {}
    signal_values    = {}
    signal_waveforms = {}
    clock_id         = None

    in_dumpvars  = False
    current_time = 0

    with open(vcd_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # -----------------------------------------------------------------
            # Capture VCD $var definitions to build a map from ID -> signal_name
            # -----------------------------------------------------------------
            if line.startswith("$var"):
                parts = line.split()
                if len(parts) >= 5:
                    sig_id = parts[3]
                    name_tokens = parts[4:-1]
                    sig_name = " ".join(name_tokens).strip()

                    signal_map[sig_id] = sig_name
                    signal_values[sig_id] = 'x'
                    if sig_name == clock_signal_name:
                        clock_id = sig_id

                    signal_waveforms[sig_name] = []
                continue

            if line.startswith("$dumpvars"):
                in_dumpvars = True
                continue
            if line.startswith("$end") and in_dumpvars:
                in_dumpvars = False
                continue

            # -----------------------------------------------------------------
            # Time markers: #0, #10, #100, etc.
            # -----------------------------------------------------------------
            if line.startswith('#'):
                try:
                    current_time = int(line[1:])
                except ValueError:
                    current_time = 0
                continue

            # -----------------------------------------------------------------
            # Value changes: single-bit or multi-bit lines
            # -----------------------------------------------------------------
            if in_dumpvars or (current_time >= 0):
                # Single-bit: "0A", "1A", "xA"
                m_single = re.match(r'([01xXzZ])(\S+)', line)
                if m_single:
                    val, sid = m_single.groups()
                    if sid in signal_values:
                        val = val.lower()
                        val = '0' if (val == 'x' or val == 'z') else val
                        signal_values[sid] = val

                        # If this is the clock toggling, record a new "sample"
                        if sid == clock_id:
                            # For every known signal, push current val into waveform
                            for s_id, curr_val in signal_values.items():
                                s_name = signal_map[s_id]
                                int_val = 1 if curr_val == '1' else 0
                                signal_waveforms[s_name].append(int_val)
                else:
                    # Multi-bit: "b1010 A"
                    m_bus = re.match(r'b([01xXzZ]+)\s+(\S+)', line)
                    if m_bus:
                        bits_str, sid = m_bus.groups()
                        if sid in signal_values:
                            bits_str = bits_str.lower().replace('x','0').replace('z','0')
                            signal_values[sid] = bits_str

                            # If this is the clock toggling, record a new "sample"
                            if sid == clock_id:
                                for s_id, curr_val in signal_values.items():
                                    s_name = signal_map[s_id]
                                    curr_val = curr_val.lower().replace('x','0').replace('z','0')
                                    int_val = int(curr_val, 2)
                                    signal_waveforms[s_name].append(int_val)

    return signal_waveforms

###############################################################################
# 2) Parallel parsing of multiple VCDs
###############################################################################
def parse_vcd_file_wrapper(vcd_file, clock_signal):
    """
    Helper for parallel usage. Calls parse_vcd_signals_per_clock on one file.
    """
    return parse_vcd_signals_per_clock(vcd_file, clock_signal)

def parse_all_vcds_in_parallel(folder, clock_signal, total_files=600):
    """
    Spawn parallel tasks to parse each vcdN.vcd.
    Returns a list of dictionaries, one per file (in order).
    """
    filepaths = [os.path.join(folder, f"vcd{i}.vcd") for i in range(1, total_files + 1)]
    all_results = [None]*total_files

    with ProcessPoolExecutor() as executor:
        futures_map = {}
        for i, path in enumerate(filepaths):
            fut = executor.submit(parse_vcd_file_wrapper, path, clock_signal)
            futures_map[fut] = i

        for fut in tqdm(as_completed(futures_map), total=total_files, desc="Parsing VCDs"):
            i = futures_map[fut]
            try:
                result = fut.result()
                all_results[i] = result
            except Exception as e:
                print(f"Error parsing {filepaths[i]}: {e}")
                all_results[i] = {}

    return all_results

###############################################################################
# 3) Unify signals + build arrays
###############################################################################
def unify_signals_and_build_arrays(all_parsed):
    """
    all_parsed: list of dicts, each dict: { signal_name: [value0, value1, ...] }
    We unify across signals so that signals_dict[sig_name] is a 2D NumPy array
      shaped (num_traces, max_len), containing integer waveforms for that signal
      for each trace (VCD file).
    """
    num_vcds = len(all_parsed)
    all_signals = set()
    for d in all_parsed:
        all_signals.update(d.keys())
    all_signals = sorted(list(all_signals))  # stable ordering

    signals_dict = {}
    for sig_name in tqdm(all_signals, desc="Unifying signals"):
        waveforms = []
        lengths   = []
        for i in range(num_vcds):
            wave_i = all_parsed[i].get(sig_name, [])
            waveforms.append(wave_i)
            lengths.append(len(wave_i))

        max_len = max(lengths)
        arr = np.zeros((num_vcds, max_len), dtype=np.int32)

        for i in range(num_vcds):
            wave_i = waveforms[i]
            if len(wave_i) < max_len:
                if len(wave_i) > 0:
                    arr[i, :len(wave_i)] = wave_i
                if len(wave_i) > 0 and len(wave_i) < max_len:
                    print(f"Warning: file #{i+1}, signal '{sig_name}' has "
                          f"{len(wave_i)} edges < {max_len}. Padded with 0.")
            else:
                arr[i, :] = wave_i

        signals_dict[sig_name] = arr

    return signals_dict

###############################################################################
# 4) Toggle array builder
###############################################################################
def build_toggle_array(signal_array):
    """
    signal_array: shape (num_traces, num_clocks)

    Return toggles: shape (num_traces, num_clocks), where
      toggles[i, t] = +1 if signal_array[i, t] != signal_array[i, t-1], else -1
    The first column toggles[i, 0] = -1 by convention (no prior sample).
    """
    num_traces, num_clocks = signal_array.shape
    toggles = np.full((num_traces, num_clocks), -1, dtype=np.int8)

    for t in range(1, num_clocks):
        toggles[:, t] = np.where(signal_array[:, t] != signal_array[:, t-1], +1, -1)

    return toggles

###############################################################################
# 5) Dot product function (per clock sample)
###############################################################################
def compute_dot_product_per_time(toggle_array, leakage_model):
    """
    toggle_array:  shape (num_traces, num_clocks)
    leakage_model: shape (num_traces,)
    Returns an array of shape (num_clocks,), where
      out[t] = dot( toggle_array[:,t], leakage_model )
    """
    num_traces, num_clocks = toggle_array.shape
    dot_products = np.zeros(num_clocks, dtype=float)

    for t in range(num_clocks):
        column = toggle_array[:, t]  # shape: (num_traces,)
        dot_products[t] = np.dot(column, leakage_model)

    return dot_products

###############################################################################
# 6) Example S-box + function to build a "bit-level HD model"
###############################################################################
sbox = [
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16
]

def extract_leakage_model_hd_bit(plaintexts, key_byte, selected_bit=0):
    """
    Simple example: for each plaintext, compute the bit-level Hamming Distance
    between the input bit and output bit of S-box.
    Return an array of +1 or -1.
    """
    leakage_model = []
    for pt in plaintexts:
        sbox_in  = pt ^ key_byte
        bit_in   = (sbox_in >> selected_bit) & 0x1
        sbox_out = sbox[sbox_in]
        bit_out  = (sbox_out >> selected_bit) & 0x1
        hd_bit   = bit_in ^ bit_out
        leakage_model.append(+1 if hd_bit == 1 else -1)
    return np.array(leakage_model, dtype=np.float64)

###############################################################################
# 7) Main usage
###############################################################################
if __name__ == "__main__":
    # ------------------ (A) Setup ------------------
    vcd_folder = "/media/abish/Extreme Pro/Sbox_600/vcdssbox"#vcdssbox"
    clock_name = "SYSCLK_P"
    total_vcds = 600

    # Example: /path/to/plaintexts.txt
    txt_path = "/media/abish/Extreme Pro/Sbox_600/plaintexts.txt"
    # The plaintexts file is assumed to have lines like:
    #   #0 : 00 11 22 33 ...
    #   #1 : aa bb cc dd ...
    # so we parse after the colon for hex values.

    # ------------------ (B) Load plaintexts ------------------
    hex_arrays = []
    with open(txt_path, "r") as file:
        for line in file:
            parts = line.strip().split(":")
            if len(parts) == 2:
                hex_values = parts[1].strip().split()
                hex_arrays.append(hex_values)

    # Convert to 2D array (num_files, bytes_per_line)
    textin_array = []
    for row in hex_arrays:
        textin_array.append([int(x, 16) for x in row])
    textin_array = np.array(textin_array, dtype=np.uint8)
    print("textin_array shape:", textin_array.shape)

    # Example AES key
    corkey = [0x2b,0x7e,0x15,0x16,0x28,0xae,0xd2,0xa6,
              0xab,0xf7,0x15,0x88,0x09,0xcf,0x4f,0x3c]
    selected_byte = 0
    chosen_bit    = 0  # LSB
    key_guess     = corkey[selected_byte]

    # Build the leakage model (shape: (num_traces,))
    plaintext_for_each_trace = textin_array[:, selected_byte]
    leakage_model = extract_leakage_model_hd_bit(plaintext_for_each_trace, key_guess, selected_bit=chosen_bit)
    print(f"Built HD model for {total_vcds} traces (bit={chosen_bit}).")

    # ------------------ (C) Parse VCDs ------------------
    print(f"Parsing {total_vcds} VCD files in parallel ...")
    all_parsed = parse_all_vcds_in_parallel(vcd_folder, clock_signal=clock_name, total_files=total_vcds)
    print("Done parsing.\n")

    # ------------------ (D) Unify signals into arrays ------------------
    print("Unifying signals across all VCDs and building arrays...")
    signals_dict = unify_signals_and_build_arrays(all_parsed)
    print(f"Done. Found {len(signals_dict)} unique signals.\n")

    # ------------------ (E) For each signal, build toggles & compute dot product ------------------
    toggle_results = []
    for sig_name, signal_array in tqdm(signals_dict.items(), desc="Toggle dot products"):
        # 1) Build toggles: shape (num_traces, num_clocks)
        toggle_array = build_toggle_array(signal_array)

        # 2) Compute dot product vs. leakage model for each clock
        dp_over_time = compute_dot_product_per_time(toggle_array, leakage_model)

        # 3) Find max absolute dot product and its clock index

        best_idx = np.argmax(np.abs(dp_over_time))
        max_val  = dp_over_time[best_idx]
        toggle_results.append((sig_name, dp_over_time, max_val, best_idx))

    # Sort by max absolute dot product
    toggle_results.sort(key=lambda x: x[2], reverse=True)

    # ------------------ (F) Print top 20 signals ------------------
    print("\n=== Top Toggle-Dot-Product Signals ===")
    for i, (sig_name, dp_arr, max_val, best_idx) in enumerate(toggle_results[:20]):
        print(f"{i+1:2d}) {sig_name}: max abs dot product={max_val:.4f} at clock={best_idx}")

    # ------------------ (G) Optional: Plot some signals ------------------
    # Example: plot top 1
    # if toggle_results:
    #     top_sig, top_dp, _, _ = toggle_results[0]
    #     plt.figure()
    #     plt.plot(top_dp)
    #     plt.title(f"Dot Product Over Time for {top_sig}")
    #     plt.xlabel("Clock sample")
    #     plt.ylabel("Dot product")
    #     plt.show()


In [ ]:
import os
import re
import time
import warnings
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
import matplotlib.pyplot as plt

###############################################################################
# 1) Single VCD parser
###############################################################################
def parse_vcd_signals_per_clock(vcd_file, clock_signal_name):
    """
    Parse a single VCD file, capturing integer values of all signals
    at each rising edge of 'clock_signal_name'.
    Returns a dict: { signal_name: [val_per_clock, ...], ... }
    """
    signal_map       = {}
    signal_values    = {}
    signal_waveforms = {}
    clock_id         = None

    in_dumpvars  = False
    current_time = 0

    with open(vcd_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Capture $var definitions => ID -> signal_name
            if line.startswith("$var"):
                parts = line.split()
                if len(parts) >= 5:
                    sig_id   = parts[3]
                    name_tok = parts[4:-1]
                    sig_name = " ".join(name_tok).strip()

                    signal_map[sig_id]       = sig_name
                    signal_values[sig_id]    = 'x'
                    signal_waveforms[sig_name] = []

                    if sig_name == clock_signal_name:
                        clock_id = sig_id
                continue

            if line.startswith("$dumpvars"):
                in_dumpvars = True
                continue
            if line.startswith("$end") and in_dumpvars:
                in_dumpvars = False
                continue

            # Time markers (#0, #100, etc.)
            if line.startswith('#'):
                try:
                    current_time = int(line[1:])
                except ValueError:
                    current_time = 0
                continue

            # Value changes
            if in_dumpvars or (current_time >= 0):
                # Single-bit: "0A", "1A", "xA"
                m_single = re.match(r'([01xXzZ])(\S+)', line)
                if m_single:
                    val, sid = m_single.groups()
                    if sid in signal_values:
                        val = val.lower()
                        val = '0' if val in ('x', 'z') else val
                        signal_values[sid] = val

                        # If the clock just toggled, record new sample for all signals
                        if sid == clock_id:
                            for s_id, curr_val in signal_values.items():
                                s_name = signal_map[s_id]
                                int_val = 1 if curr_val == '1' else 0
                                signal_waveforms[s_name].append(int_val)

                else:
                    # Multi-bit: "b1010 A"
                    m_bus = re.match(r'b([01xXzZ]+)\s+(\S+)', line)
                    if m_bus:
                        bits_str, sid = m_bus.groups()
                        if sid in signal_values:
                            bits_str = bits_str.lower().replace('x','0').replace('z','0')
                            signal_values[sid] = bits_str

                            if sid == clock_id:
                                for s_id, curr_val in signal_values.items():
                                    s_name   = signal_map[s_id]
                                    curr_val = curr_val.lower().replace('x','0').replace('z','0')
                                    int_val  = int(curr_val, 2)
                                    signal_waveforms[s_name].append(int_val)

    return signal_waveforms


###############################################################################
# 2) Parallel parse for all VCD files
###############################################################################
def parse_vcd_file_wrapper(args):
    """
    Helper for parallel usage. We pass a tuple of (vcd_file, clock_signal).
    """
    vcd_file, clock_signal = args
    return parse_vcd_signals_per_clock(vcd_file, clock_signal)


def parse_all_vcds_in_parallel(folder, clock_signal, total_files=600):
    """
    Spin up parallel tasks to parse each file vcdN.vcd.
    Returns a list of dictionaries, one per file index.
    """
    filepaths = [os.path.join(folder, f"vcd{i}.vcd") for i in range(1, total_files + 1)]
    all_results = [None]*total_files

    tasks = [(filepaths[i], clock_signal) for i in range(total_files)]

    with ProcessPoolExecutor() as executor:
        futures_map = {executor.submit(parse_vcd_file_wrapper, t): idx for idx, t in enumerate(tasks)}

        for fut in tqdm(as_completed(futures_map), total=total_files, desc="Parsing VCDs"):
            i = futures_map[fut]
            try:
                res = fut.result()
                all_results[i] = res
            except Exception as e:
                print(f"[!] Error parsing {filepaths[i]}: {e}")
                all_results[i] = {}

    return all_results


###############################################################################
# 3) Unify signals
###############################################################################
def unify_signals_and_build_arrays(all_parsed):
    """
    Given a list of dicts (one per VCD), unify so that signals_dict[sig_name]
    is a 2D np.array of shape (num_traces, max_len).
    """
    num_vcds = len(all_parsed)
    all_signals = set()
    for d in all_parsed:
        all_signals.update(d.keys())
    all_signals = sorted(list(all_signals))

    signals_dict = {}
    for sig_name in tqdm(all_signals, desc="Unifying signals"):
        waveforms = []
        lengths = []
        for i in range(num_vcds):
            wave_i = all_parsed[i].get(sig_name, [])
            waveforms.append(wave_i)
            lengths.append(len(wave_i))

        max_len = max(lengths)
        arr = np.zeros((num_vcds, max_len), dtype=np.int32)

        for i in range(num_vcds):
            wave_i = waveforms[i]
            if len(wave_i) > 0:
                arr[i, :len(wave_i)] = wave_i
            if len(wave_i) < max_len and len(wave_i) > 0:
                print(f"[!] Warning: file {i+1}, signal '{sig_name}' has "
                      f"{len(wave_i)} edges < {max_len}. Padded with 0.")

        signals_dict[sig_name] = arr

    return signals_dict


###############################################################################
# 4) Build toggle array
###############################################################################
def build_toggle_array(signal_array):
    """
    Given signal_array shape (num_traces, num_clocks),
    produce toggles of the same shape: +1 if toggles from previous clock, else -1.
    toggles[:, 0] = -1 by convention.
    """
    num_traces, num_clocks = signal_array.shape
    toggles = np.full((num_traces, num_clocks), -1, dtype=np.int8)

    for t in range(1, num_clocks):
        toggles[:, t] = np.where(signal_array[:, t] != signal_array[:, t - 1], +1, -1)

    return toggles


###############################################################################
# 5) Dot product at each clock sample
###############################################################################
def compute_dot_product_per_time(toggle_array, leakage_model):
    """
    toggle_array: shape (num_traces, num_clocks)
    leakage_model: shape (num_traces,)
    Return dp[t] = sum(toggle_array[i, t] * leakage_model[i]) for i in [0..num_traces-1].
    """
    num_traces, num_clocks = toggle_array.shape
    dot_products = np.zeros(num_clocks, dtype=float)

    # vectorized approach:
    # dot_products = toggle_array.T @ leakage_model
    # but let's keep the loop for clarity:
    for t in range(num_clocks):
        column = toggle_array[:, t]
        dot_products[t] = np.dot(column, leakage_model)

    return dot_products


###############################################################################
# 6) Example: HD model for a single bit
###############################################################################
sbox = [
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16
]

def extract_leakage_model_hd_bit(plaintexts, key_byte, selected_bit=0):
    """
    For each plaintext, compute the HD between the input bit and output bit of S-box.
    Return array of +1 / -1.
    """
    model = []
    for pt in plaintexts:
        sbox_in  = pt ^ key_byte
        bit_in   = (sbox_in >> selected_bit) & 0x1
        sbox_out = sbox[sbox_in]
        bit_out  = (sbox_out >> selected_bit) & 0x1
        hd_bit   = bit_in ^ bit_out
        model.append(+1 if hd_bit == 1 else -1)
    return np.array(model, dtype=np.float64)


###############################################################################
# 7) Function to compute toggle dot-products for a given key guess
###############################################################################
def compute_toggle_dot_for_key_guess(key_guess, plaintexts, toggles_dict):
    """
    key_guess: int in [0..127] or [0..255], etc.
    plaintexts: shape (num_traces,)
    toggles_dict: { signal_name: 2D toggle array [num_traces, num_clocks] }

    Returns:
      ( key_guess,
        list of (sig_name, max_abs_dot, best_clock_idx),
        iteration_runtime_seconds )
    """
    start_time = time.time()

    # 1) Build the leakage model
    #   For demonstration, we fix selected_bit=0 (LSB)
    #   If you want multi-bit or multiple bytes, adapt accordingly
    selected_bit = 0
    leakage_model = extract_leakage_model_hd_bit(plaintexts, key_guess, selected_bit=selected_bit)

    # 2) For each signal, compute the max absolute dot across time
    results_for_signals = []
    for sig_name, toggle_array in toggles_dict.items():
        dp_over_time = compute_dot_product_per_time(toggle_array, leakage_model)
        max_val  = np.max(np.abs(dp_over_time))
        best_idx = np.argmax(np.abs(dp_over_time))
        results_for_signals.append((sig_name, max_val, best_idx))

    # You might only want to store the top 1 or top N signals to save memory.
    # For demonstration, let's keep them all. If large, consider truncating.

    end_time = time.time()
    iteration_time = end_time - start_time

    return (key_guess, results_for_signals, iteration_time)


###############################################################################
# 8) Main
###############################################################################
if __name__ == "__main__":
    # ----------------- (A) Paths and Setup -----------------
    vcd_folder = "/media/abish/Extreme Pro/Sbox_600/vcdssbox"#vcdssbox"
    clock_name = "SYSCLK_P"
    total_vcds = 600

    # Example: /path/to/plaintexts.txt
    txt_path = "/media/abish/Extreme Pro/Sbox_600/plaintexts.txt"
    hex_arrays = []
    with open(txt_path, "r") as file:
        for line in file:
            parts = line.strip().split(":")
            if len(parts) == 2:
                hex_values = parts[1].strip().split()
                hex_arrays.append(hex_values)

    # Convert to 2D array (num_vcds, bytes_per_line)
    textin_array = []
    for row in hex_arrays:
        textin_array.append([int(x, 16) for x in row])
    textin_array = np.array(textin_array, dtype=np.uint8)
    print("textin_array shape:", textin_array.shape)

    # In this example, let's only brute-force key guesses in [0..127].
    # For full AES keyspace of one byte, do range(256).
    possible_key_guesses = range(128)

    # We'll use only one AES byte (e.g., byte 0).
    selected_byte_index = 0
    # plaintext_for_each_trace => shape (600,) if we have 600 VCDs
    plaintext_for_each_trace = textin_array[:, selected_byte_index]

    # ----------------- (B) Parse VCDs in parallel -----------------
    parse_start = time.time()
    all_parsed = parse_all_vcds_in_parallel(vcd_folder, clock_signal=clock_name, total_files=total_vcds)
    parse_end = time.time()
    print(f"[Timing] Parsing all VCDs took {parse_end - parse_start:.2f} s.")

    # ----------------- (C) Unify signals -----------------
    unify_start = time.time()
    signals_dict = unify_signals_and_build_arrays(all_parsed)
    unify_end = time.time()
    print(f"[Timing] Unifying signals took {unify_end - unify_start:.2f} s.")
    print(f"Found {len(signals_dict)} unique signals.")

    # ----------------- (D) Pre-build toggles for each signal -----------------
    #  This does NOT depend on the key guess => we do it once.
    toggle_start = time.time()
    toggles_dict = {}
    for sig_name, sig_array in tqdm(signals_dict.items(), desc="Build toggles"):
        toggles_dict[sig_name] = build_toggle_array(sig_array)
    toggle_end = time.time()
    print(f"[Timing] Building toggles took {toggle_end - toggle_start:.2f} s.")

    # ----------------- (E) For each key guess, compute dot-product in parallel -----------------
    def parallel_key_guess_wrapper(k):
        # We pass the minimal data needed: the single plaintext column + toggles
        return compute_toggle_dot_for_key_guess(k, plaintext_for_each_trace, toggles_dict)

    all_keys_start = time.time()
    iteration_times = []

    # Option A) PARALLEL approach across key guesses
    with ProcessPoolExecutor() as executor:
        futures = [executor.submit(parallel_key_guess_wrapper, k) for k in possible_key_guesses]
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Key guesses"):
            try:
                key_guess, results_for_signals, iter_time = fut.result()
                iteration_times.append(iter_time)
                # Optionally store or print partial results:
                # e.g. find top signals for this key guess
                # sorted_signals = sorted(results_for_signals, key=lambda x: x[1], reverse=True)
                # top_signal_name, top_val, top_idx = sorted_signals[0]
                # print(f"Key 0x{key_guess:02X}: best signal={top_signal_name} val={top_val} clock={top_idx}")
            except Exception as e:
                print(f"[!] Error in key guess job: {e}")

    # Option B) SERIAL approach (if you prefer to measure iteration time easily):
    # all_results = []
    # for k in tqdm(possible_key_guesses, desc="Key guesses"):
    #     r = parallel_key_guess_wrapper(k)
    #     all_results.append(r)
    #     iteration_times.append(r[-1])

    all_keys_end = time.time()
    total_time_for_all_keys = all_keys_end - all_keys_start

    # ----------------- (F) Print timing statistics -----------------
    print("\n--- TIMING SUMMARY ---")
    print(f"VCD parsing time:   {parse_end - parse_start:.2f} s")
    print(f"Unify signals time: {unify_end - unify_start:.2f} s")
    print(f"Build toggles time: {toggle_end - toggle_start:.2f} s")

    print(f"\nKey guess computations: {len(possible_key_guesses)} total")
    print(f"Total time for all keys: {total_time_for_all_keys:.2f} s")

    if iteration_times:
        avg_iter = np.mean(iteration_times)
        min_iter = np.min(iteration_times)
        max_iter = np.max(iteration_times)
        print(f"  Per-iteration stats (in parallel):")
        print(f"    Average = {avg_iter:.4f} s")
        print(f"    Min     = {min_iter:.4f} s")
        print(f"    Max     = {max_iter:.4f} s")

    # -------------- Optional: do something with the final results --------------
    # In the parallel approach above, you might have stored partial results in a global or
    # appended them to a manager structure. Or you can switch to the serial approach if you
    # want a local all_results list to analyze further.


In [ ]:
from tqdm import tqdm  # Add this at the top

def analyze_high_correlation_signals(ranked_signals, netlist_file, tech_file, power_report_file):
    gates_dict = parse_gate_definitions(tech_file)
    power_data = extract_power_per_instance(power_report_file)
    signal_gate_mapping = {}

    for (signal, dp_arr, correlation, best_idx)  in tqdm(ranked_signals, desc="Analyzing Signals", unit="sig"):
        nesi = False
        #if abs(correlation) <= analysis_correlation_limitation:
           # continue

        processed_signal = preprocess_signal_name(signal)
        connections = find_signal_in_netlist(processed_signal, netlist_file)

        signal_gate_mapping[signal] = {
            "correlation": correlation,
            "gates": [],
            "Impact": 0
        }

        instance_powers = []
        for connection in connections:
            gate_matchxc = re.search(r'(SC8T_\S+)\s+(\S+)\(', connection)
            if gate_matchxc is None:
                gate_matchxc = re.search(r'(SC8T_\S+)\s+(\S+)\s*\(', connection)

            if gate_matchxc:
                gate_name = gate_matchxc.group(1)
                instance_namef = gate_matchxc.group(2)
                instance_name = instance_namef.replace("\\", "")

                if gate_name in gates_dict:
                    gate_info = gates_dict[gate_name]
                    port_signal_map = parse_connection_ports(connection)
                    roles = []

                    for port, s_name in port_signal_map.items():
                        if s_name == processed_signal:
                            if port in gate_info["inputs"]:
                                roles.append("input")
                            if port in gate_info["outputs"]:
                                roles.append("output")

                    if roles:
                        signal_gate_mapping[signal]["gates"].append({
                            "gate_name": gate_name,
                            "gate_info": gate_info,
                            "instance_name": instance_name,
                            "connection": connection,
                            "roles": roles
                        })

                        if "output" in roles and not nesi:
                            for power_entry in power_data.keys():
                                power_instance_name = power_entry.split('/')[-1]
                                if instance_name == power_instance_name:
                                    instance_powers.append(power_data[power_entry])
                                    nesi = True
                                    break

        if instance_powers:
            average_power = sum(instance_powers) / len(instance_powers)
            signal_gate_mapping[signal]["Impact"] = average_power / total_Power

    return signal_gate_mapping


## Netlist process (get gates accourding to signal) + compute Impact factor
import re

analysis_correlation_limitation = 1
total_Power = 1.39427e-04


# Preprocess signal names
def preprocess_signal_name(signal):
    return signal.replace("][", "] [")


# Calculate the ranking based on Correlation x Average Power
def rank_signals_by_impact(signal_gate_mapping):
    ranked_signals = []

    for signal, data in signal_gate_mapping.items():
        # Skip signals with no calculated impact
        #if data["Impact"] > 0:
        # Handle NaN correlation by assuming it as 0
        try:
            correlation = data["correlation"]
            if correlation is None:
                correlation = 0  # Treat NaN as 0

                # Calculate the impact value
            impact_value = correlation * data["Impact"]

            # Find the gate instance name with the "output" role
            output_instance = None
            for gate in data["gates"]:
                if "output" in gate["roles"]:
                    output_instance = gate["instance_name"]
                    break

            # Add the signal, impact value, and output instance to the ranking list
            ranked_signals.append({
                "signal": signal,
                "impact_value": impact_value,
                "output_instance": output_instance,
                "correlation": correlation,  # Include correlation for debugging purposes
            })
        except KeyError as e:
            print(f"KeyError for signal '{signal}': Missing key {e}")
        except Exception as e:
            print(f"Unexpected error for signal '{signal}': {e}")
    # Sort signals by impact value in descending order
    ranked_signals.sort(key=lambda x: x["impact_value"], reverse=True)

    return ranked_signals


# Debugging Output: Verify Intermediate Data
def print_debug_data(signal_gate_mapping):
    print("\nSignal Data (Before Ranking):")
    for signal, data in signal_gate_mapping.items():
        print(f"Signal: {signal}, Correlation: {data['correlation']}, Impact: {data['Impact']:.6e}")
        for gate in data["gates"]:
            print(f"  Gate: {gate['gate_name']}, Instance: {gate['instance_name']}, Role: {gate['role']}")


# Debugging Output: Verify Ranked Signals
def print_ranked_signals(ranked_signals):
    print("\nRanked Signals by Impact Value:")
    for rank, entry in enumerate(ranked_signals, start=1):
        print(f"Rank {rank}: Signal = {entry['signal']}, "
              f"Impact Value = {entry['impact_value']:.6e}, "
              f"correlation = {entry['correlation']}, "
              f"Output Instance = {entry['output_instance']}")


def find_signal_in_netlist(signal, netlist_file):
    connections = []
    with open(netlist_file, 'r') as file:
        lines = file.readlines()

    # Create a regex pattern to match the exact signal name
    escaped_signal = re.escape(signal).replace(r'\[', r'\s*\[').replace(r'\]', r'\s*\]')
    signal_pattern = re.compile(rf'(?<!\w){escaped_signal}(?!\w)')  # Match the signal exactly

    for i, line in enumerate(lines):
        # Skip wire declarations
        if line.strip().startswith("wire"):
            continue

        # Check if the signal is in the current line
        if signal_pattern.search(line):  # Use regex search for exact match
            # Backtrack to find the start of the gate (starting with SC8T_)
            start_idx = i
            while start_idx > 0 and not lines[start_idx].strip().startswith("SC8T_"):
                start_idx -= 1

            # Forward track to find the end of the gate (ending with ;)
            end_idx = i
            while end_idx < len(lines) and ";" not in lines[end_idx]:
                end_idx += 1

            # Capture the full gate definition
            gate_definition = "".join(lines[start_idx:end_idx + 1]).strip()
            gate_definition_cleaned = re.sub(r'\s+', ' ', gate_definition)

            connections.append(gate_definition_cleaned)

    return connections


# Parse gate definitions from the technology file
def parse_gate_definitions(tech_file):
    gates_dict = {}
    module_def_regex = re.compile(r'module\s+(\S+)\s*\(([^)]*)\);')
    input_regex = re.compile(r'input\s+([^;]+);')
    output_regex = re.compile(r'output\s+([^;]+);')

    with open(tech_file, 'r') as vf:
        verilog_content = vf.read()

    for match in module_def_regex.finditer(verilog_content):
        gate_name, ports = match.groups()
        ports = [p.strip() for p in ports.split(',')]

        # Extract inputs and outputs
        inputs = []
        outputs = []
        start_idx = verilog_content.find(match.group(0)) + len(match.group(0))
        module_body = verilog_content[start_idx:verilog_content.find("endmodule", start_idx)]

        for input_match in input_regex.finditer(module_body):
            inputs.extend(input_match.group(1).split(','))

        for output_match in output_regex.finditer(module_body):
            outputs.extend(output_match.group(1).split(','))

        gates_dict[gate_name] = {
            "name": gate_name,
            "inputs": [i.strip() for i in inputs],
            "outputs": [o.strip() for o in outputs],
            "ports": ports
        }
    return gates_dict


# Parse port-to-signal mapping
def parse_connection_ports(connection):
    port_signal_regex = re.findall(r'\.(\w+)\s*\(\s*(.*?)\s*\)', connection)
    return {port: signal.replace("\\", "").strip() for port, signal in port_signal_regex}


# Extract power for each instance from the power report
def extract_power_per_instance(power_report_file):
    power_per_instance = {}
    with open(power_report_file, 'r') as file:
        for line in file:
            if re.match(r'^\s*[0-9eE.+-]+\s+[0-9eE.+-]+\s+[0-9eE.+-]+\s+[0-9eE.+-]+\s+/.+', line):
                parts = line.split()
                total_power = float(parts[-2])  # Extract total power
                instance_name = parts[-1]  # Extract instance name
                power_per_instance[instance_name] = total_power
    return power_per_instance



# Example Usage
if __name__ == "__main__":
    ranked_signals_new = []
    path = "/media/abish/Extreme Pro/sbox_600"
    netlist_file = path + "/" + "PROACT_topa_netlist.v"
    tech_file = path + "/" + "GF22FDX_SC8T_116CPP_BASE_DDC28UH.v"
    power_report_file = path + "/" + "Power_per_instance.txt"

    signal_gate_mapping = analyze_high_correlation_signals(toggle_results, netlist_file, tech_file,
                                                           power_report_file)

    # Print results
    ranked_signals_by_impact = rank_signals_by_impact(signal_gate_mapping)

    print_ranked_signals(ranked_signals_by_impact)

In [ ]:
import matplotlib.pyplot as plt

# Example: Extract impact values from ranked_signals_by_impact (replace this with your actual data)
impact_values = [entry["impact_value"] for entry in ranked_signals_by_impact]

# Sort impact values
impact_values = sorted(impact_values)

# Plotting the Histogram
plt.figure(figsize=(10, 6))
plt.hist(impact_values, bins=50, log=True, color='blue', edgecolor='black')
plt.xlabel('Leakage Impact Factor (LIF)', fontsize=12)
plt.ylabel('Number of Cells', fontsize=12)
plt.title('Leakage Impact Factor Distribution', fontsize=14)

# Highlight and Annotate Top Ranking Cells
top_3 = sorted(impact_values, reverse=True)[:3]  # Top 3 impact values
positions = [1, 2, 3]  # For ranking (1st, 2nd, 3rd)
for i, value in enumerate(top_3):
    plt.axvline(value, color='black', linestyle='--', linewidth=1)
    plt.text(value, 10, f"{positions[i]} Ranking Cell", rotation=90, fontsize=10, ha='right', va='bottom')

# Adjust the plot
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.tight_layout()
plt.savefig("NETLIF_distribution.pdf", bbox_inches="tight", transparent=True)
plt.savefig("NETLIF_correlation_distribution.svg", bbox_inches="tight", transparent=True)
plt.savefig("NETLIF_correlation_distribution.png", bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
import pandas as pd

# Define LIF ranges
lif_ranges = [
    (0.12, 0.2),
    (0.07, 0.12),
    (0.03, 0.07),
    (-0.2, 0.03),
]

# Initialize a dictionary to store the counts for each range
lif_distribution = {"LIF Range": [], "No. of Cells": []}

# Iterate through the defined ranges
for lif_min, lif_max in lif_ranges:
    # Count signals within the current range
    count = sum(
        lif_min <= entry["impact_value"] <= lif_max
        for entry in ranked_signals_by_impact
    )
    lif_distribution["LIF Range"].append(f"{lif_min} ~ {lif_max}")
    lif_distribution["No. of Cells"].append(count)

# Convert to DataFrame for a table-like representation
df_lif = pd.DataFrame(lif_distribution)

# Print the table
print("TABLE III: LIF Distribution Data")
print(df_lif.to_string(index=False))

# Optionally save the table to a CSV
df_lif.to_csv("table_III_LIF_distribution.csv", index=False)


In [ ]:
## New mthode test here

## Netlist process (get gates accourding to signal) + compute Impact factor

In [ ]:
import re
analysis_correlation_limitation = 10
total_Power = 9.28787e-05
# Preprocess signal names
def preprocess_signal_name(signal):
    return signal.replace("][", "] [")
# Calculate the ranking based on Correlation x Average Power
def rank_signals_by_impact(signal_gate_mapping):
    ranked_signals = []

    for signal, data in signal_gate_mapping.items():
        # Skip signals with no calculated impact
        #if data["Impact"] > 0:
            # Handle NaN correlation by assuming it as 0
        correlation = data["correlation"]
        if correlation is None:
            correlation = 0  # Treat NaN as 0

            # Calculate the impact value
        impact_value = correlation * data["Impact"]

            # Find the gate instance name with the "output" role
        output_instance = None
        for gate in data["gates"]:
            if gate["role"] == "output":
                output_instance = gate["instance_name"]
                break

            # Add the signal, impact value, and output instance to the ranking list
        ranked_signals.append({
                "signal": signal,
                "impact_value": impact_value,
                "output_instance": output_instance,
                "correlation": correlation,  # Include correlation for debugging purposes
            })

    # Sort signals by impact value in descending order
    ranked_signals.sort(key=lambda x: x["impact_value"], reverse=True)

    return ranked_signals

# Debugging Output: Verify Intermediate Data
def print_debug_data(signal_gate_mapping):
    print("\nSignal Data (Before Ranking):")
    for signal, data in signal_gate_mapping.items():
        print(f"Signal: {signal}, Correlation: {data['correlation']}, Impact: {data['Impact']:.6e}")
        for gate in data["gates"]:
            print(f"  Gate: {gate['gate_name']}, Instance: {gate['instance_name']}, Role: {gate['role']}")

# Debugging Output: Verify Ranked Signals
def print_ranked_signals(ranked_signals):
    print("\nRanked Signals by Impact Value:")
    for rank, entry in enumerate(ranked_signals, start=1):
        print(f"Rank {rank}: Signal = {entry['signal']}, "
              f"Impact Value = {entry['impact_value']:.6e}, "
              f"correlation = {entry['correlation']}, "
              f"Output Instance = {entry['output_instance']}")

def find_signal_in_netlist(signal, netlist_file):
    connections = []
    with open(netlist_file, 'r') as file:
        lines = file.readlines()

    # Create a regex pattern to match the exact signal name
    escaped_signal = re.escape(signal).replace(r'\[', r'\s*\[').replace(r'\]', r'\s*\]')
    signal_pattern = re.compile(rf'(?<!\w){escaped_signal}(?!\w)')  # Match the signal exactly

    for i, line in enumerate(lines):
        # Skip wire declarations
        if line.strip().startswith("wire"):
            continue

        # Check if the signal is in the current line
        if signal_pattern.search(line):  # Use regex search for exact match
            # Backtrack to find the start of the gate (starting with SC8T_)
            start_idx = i
            while start_idx > 0 and not lines[start_idx].strip().startswith("SC8T_"):
                start_idx -= 1

            # Forward track to find the end of the gate (ending with ;)
            end_idx = i
            while end_idx < len(lines) and ";" not in lines[end_idx]:
                end_idx += 1

            # Capture the full gate definition
            gate_definition = "".join(lines[start_idx:end_idx + 1]).strip()
            gate_definition_cleaned = re.sub(r'\s+', ' ', gate_definition)

            connections.append(gate_definition_cleaned)

    return connections

# Parse gate definitions from the technology file
def parse_gate_definitions(tech_file):
    gates_dict = {}
    module_def_regex = re.compile(r'module\s+(\S+)\s*\(([^)]*)\);')
    input_regex = re.compile(r'input\s+([^;]+);')
    output_regex = re.compile(r'output\s+([^;]+);')

    with open(tech_file, 'r') as vf:
        verilog_content = vf.read()

    for match in module_def_regex.finditer(verilog_content):
        gate_name, ports = match.groups()
        ports = [p.strip() for p in ports.split(',')]

        # Extract inputs and outputs
        inputs = []
        outputs = []
        start_idx = verilog_content.find(match.group(0)) + len(match.group(0))
        module_body = verilog_content[start_idx:verilog_content.find("endmodule", start_idx)]

        for input_match in input_regex.finditer(module_body):
            inputs.extend(input_match.group(1).split(','))

        for output_match in output_regex.finditer(module_body):
            outputs.extend(output_match.group(1).split(','))

        gates_dict[gate_name] = {
            "name": gate_name,
            "inputs": [i.strip() for i in inputs],
            "outputs": [o.strip() for o in outputs],
            "ports": ports
        }
    return gates_dict

# Parse port-to-signal mapping
def parse_connection_ports(connection):
    port_signal_regex = re.findall(r'\.(\w+)\s*\(\s*(.*?)\s*\)', connection)
    return {port: signal.replace("\\", "").strip() for port, signal in port_signal_regex}

# Extract power for each instance from the power report
def extract_power_per_instance(power_report_file):
    power_per_instance = {}
    with open(power_report_file, 'r') as file:
        for line in file:
            if re.match(r'^\s*[0-9eE.+-]+\s+[0-9eE.+-]+\s+[0-9eE.+-]+\s+[0-9eE.+-]+\s+/.+', line):
                parts = line.split()
                total_power = float(parts[-2])  # Extract total power
                instance_name = parts[-1]  # Extract instance name
                power_per_instance[instance_name] = total_power
    return power_per_instance

# Analyze high-correlation signals
def analyze_high_correlation_signals(ranked_signals, netlist_file, tech_file, power_report_file):
    gates_dict = parse_gate_definitions(tech_file)
    power_data = extract_power_per_instance(power_report_file)
    signal_gate_mapping = {}

    for signal, correlation in ranked_signals:
        if abs(correlation) <= analysis_correlation_limitation:
            continue
        processed_signal = preprocess_signal_name(signal)
        connections = find_signal_in_netlist(processed_signal, netlist_file)

        signal_gate_mapping[signal] = {
            "correlation": correlation,
            "gates": [],
            "Impact": 0
        }

        instance_powers = []
        for connection in connections:
            # Extract gate instantiation name
            gate_match = re.search(r'(SC8T_\S+)', connection)
            gate_matchxc = re.search(r'(SC8T_\S+)\s+(\S+)\(', connection)
            if gate_matchxc == None:
                gate_matchxc = re.search(r'(SC8T_\S+)\s+(\S+)\s*\(', connection)
            if gate_match:
                gate_name = gate_match.group(1)
                if gate_name in gates_dict:
                    gate_info = gates_dict[gate_name]
                    instance_namef = gate_matchxc.group(2)
                    instance_name = instance_namef.replace("\\", "")


                    # Parse port-to-signal mapping
                    port_signal_map = parse_connection_ports(connection)

                    # Determine role
                    role = "unknown"
                    for port, signal_name in port_signal_map.items():
                        if signal_name == processed_signal:
                            if port in gate_info["inputs"]:
                                role = "input"
                            elif port in gate_info["outputs"]:
                                role = "output"
                            break

                    # Append gate information
                    signal_gate_mapping[signal]["gates"].append({
                        "gate_name": gate_name,
                        "gate_info": gate_info,
                        "instance_name": instance_name,
                        "connection": connection,
                        "role": role
                    })
                    if role=="output":
                        for power_entry in power_data.keys():  # Iterate through power data entries
                            power_instance_name = power_entry.split('/')[-1]  # Get the part after the last '/'
                            if instance_name == power_instance_name:
                            # Instance name matches, retrieve the power value
                                instance_powers.append(power_data[power_entry])
                                break

                    # After collecting powers, calculate average


        # Calculate average power for the signal
        if instance_powers:
            average_power = sum(instance_powers) / len(instance_powers)  # Sum up all the instance powers
            signal_gate_mapping[signal]["Impact"] = average_power / total_Power



    return signal_gate_mapping

# Example Usage
if __name__ == "__main__":
    ranked_signals_new=[]
    netlist_file = path +"/"+"PROACT_topa_netlist.v"
    tech_file = path +"/"+"GF22FDX_SC8T_116CPP_BASE_DDC28UH.v"
    power_report_file = path +"/"+"Power_per_instance.txt"

    signal_gate_mapping = analyze_high_correlation_signals(ranked_signals, netlist_file, tech_file, power_report_file)

    # Print results
    ranked_signals_by_impact = rank_signals_by_impact(signal_gate_mapping)

    print_ranked_signals(ranked_signals_by_impact)
